In [6]:
# ============================================================
# STEP 1: Load Beneficiary Data + Build HEDIS Denominator

# ============================================================

import pandas as pd
import numpy as np

DATA_DIR = 'synpuf_sample1'  # change if your folder name differs

# ------------------------------------------------------------
# Load all 3 years of beneficiary summary files
# ------------------------------------------------------------
ben_2008 = pd.read_csv(f'{DATA_DIR}/DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv')
ben_2009 = pd.read_csv(f'{DATA_DIR}/DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv')
ben_2010 = pd.read_csv(f'{DATA_DIR}/DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv')

print(f"2008 beneficiaries: {len(ben_2008)}")
print(f"2009 beneficiaries: {len(ben_2009)}")
print(f"2010 beneficiaries: {len(ben_2010)}")

# ------------------------------------------------------------
# We'll use 2010 as our measurement year (most complete claims history behind it)
# ------------------------------------------------------------
ben = ben_2010.copy()

print("\nColumns available:")
print(ben.columns.tolist())

# ------------------------------------------------------------
# Build age as of 2010
# ------------------------------------------------------------
ben['BENE_BIRTH_DT'] = pd.to_datetime(ben['BENE_BIRTH_DT'], format='%Y%m%d')
ben['age_2010'] = 2010 - ben['BENE_BIRTH_DT'].dt.year

# Death date (if present, patient died - check if before measurement year end)
ben['BENE_DEATH_DT'] = pd.to_datetime(ben['BENE_DEATH_DT'], format='%Y%m%d', errors='coerce')
ben['died_in_2010_or_before'] = ben['BENE_DEATH_DT'].dt.year <= 2010

# ------------------------------------------------------------
# HEDIS Denominator criteria:
# 1. Diabetic (SP_DIABETES == 1, note: SynPUF codes chronic conditions as 1=yes, 2=no)
# 2. Age 18-75
# 3. Continuously enrolled (using HI + SMI coverage months as proxy, >=11 months)
# 4. Alive through measurement year
# ------------------------------------------------------------
print("\nSP_DIABETES value counts:")
print(ben['SP_DIABETES'].value_counts())

is_diabetic = ben['SP_DIABETES'] == 1
is_right_age = ben['age_2010'].between(18, 75)
is_enrolled = (ben['BENE_HI_CVRAGE_TOT_MONS'] >= 11) | (ben['BENE_SMI_CVRAGE_TOT_MONS'] >= 11)
is_alive = ~ben['died_in_2010_or_before'].fillna(False)

ben['in_denominator'] = (is_diabetic & is_right_age & is_enrolled & is_alive).astype(int)

print(f"\nTotal beneficiaries: {len(ben)}")
print(f"Diabetic: {is_diabetic.sum()}")
print(f"Diabetic + right age: {(is_diabetic & is_right_age).sum()}")
print(f"Final denominator (eligible patients): {ben['in_denominator'].sum()}")

denom_df = ben[ben['in_denominator'] == 1].copy()
print(f"\nDenominator dataframe shape: {denom_df.shape}")
print(denom_df[['DESYNPUF_ID', 'age_2010', 'BENE_SEX_IDENT_CD', 'SP_DIABETES',
                 'SP_CHF', 'SP_CHRNKIDN', 'SP_ISCHMCHT']].head(10))

# Save this intermediate result so Step 2 can pick it up
denom_df.to_csv('denominator_patients.csv', index=False)
print("\nSaved denominator_patients.csv")

2008 beneficiaries: 10000
2009 beneficiaries: 9856
2010 beneficiaries: 9710

Columns available:
['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR']

SP_DIABETES value counts:
SP_DIABETES
2    6897
1    2813
Name: count, dtype: int64

Total beneficiaries: 9710
Diabetic: 2813
Diabetic + right age: 1413
Final denominator (eligible patients): 1380

Denominator dataframe shape: (1380, 35)
         DESYNPUF_ID  age_2010  BENE_SEX_IDENT_CD  SP_DIABETES  SP_CHF  \
0   00016F745862898F        67                  1            1       1   

In [7]:
# ============================================================
# STEP 2: Find A1c Tests in Claims -> Build HEDIS Numerator
# ============================================================

import pandas as pd
import numpy as np

DATA_DIR = 'synpuf_sample1'

# ------------------------------------------------------------
# Load denominator patients from Step 1
# ------------------------------------------------------------
denom_df = pd.read_csv('denominator_patients.csv')
denom_ids = set(denom_df['DESYNPUF_ID'])
print(f"Denominator patients to check: {len(denom_ids)}")

# ------------------------------------------------------------
# A1c test HCPCS codes (CPT codes for HbA1c lab test)
# 83036 = Hemoglobin A1c
# 83037 = Hemoglobin A1c by device cleared by FDA for home use
# ------------------------------------------------------------
A1C_CODES = {'83036', '83037'}
HCPCS_COLS = [f'HCPCS_CD_{i}' for i in range(1, 14)]

# ------------------------------------------------------------
# Carrier claims are split into 1A and 1B - load both, in chunks
# (these files are large, ~100MB each, so we only keep relevant columns)
# ------------------------------------------------------------
USECOLS = ['DESYNPUF_ID', 'CLM_FROM_DT'] + HCPCS_COLS

def find_a1c_claims(filepath, denom_ids, chunksize=50000):
    """Stream through a large carrier claims file, keep only rows for
    denominator patients that contain an A1c HCPCS code."""
    matches = []
    for chunk in pd.read_csv(filepath, usecols=USECOLS, dtype=str, chunksize=chunksize):
        # Only look at our diabetic denominator patients
        chunk = chunk[chunk['DESYNPUF_ID'].isin(denom_ids)]
        if chunk.empty:
            continue
        # Check if ANY of the 13 HCPCS columns contain an A1c code
        has_a1c = chunk[HCPCS_COLS].isin(A1C_CODES).any(axis=1)
        if has_a1c.any():
            matches.append(chunk[has_a1c][['DESYNPUF_ID', 'CLM_FROM_DT']])
    if matches:
        return pd.concat(matches, ignore_index=True)
    return pd.DataFrame(columns=['DESYNPUF_ID', 'CLM_FROM_DT'])

print("\nScanning Carrier Claims Sample 1A for A1c tests")
a1c_1a = find_a1c_claims(f'{DATA_DIR}/DE1_0_2008_to_2010_Carrier_Claims_Sample_1A.csv', denom_ids)
print(f"Found {len(a1c_1a)} A1c claim lines in file 1A")

print("\nScanning Carrier Claims Sample 1B for A1c tests...")
a1c_1b = find_a1c_claims(f'{DATA_DIR}/DE1_0_2008_to_2010_Carrier_Claims_Sample_1B.csv', denom_ids)
print(f"Found {len(a1c_1b)} A1c claim lines in file 1B")

a1c_claims = pd.concat([a1c_1a, a1c_1b], ignore_index=True)
print(f"\nTotal A1c claim lines found: {len(a1c_claims)}")

# ------------------------------------------------------------
# Filter to measurement year 2010, build numerator flag
# ------------------------------------------------------------
a1c_claims['CLM_FROM_DT'] = pd.to_datetime(a1c_claims['CLM_FROM_DT'], format='%Y%m%d')
a1c_claims['claim_year'] = a1c_claims['CLM_FROM_DT'].dt.year

a1c_2010 = a1c_claims[a1c_claims['claim_year'] == 2010]
compliant_ids = set(a1c_2010['DESYNPUF_ID'].unique())
print(f"\nUnique patients with >=1 A1c test in 2010: {len(compliant_ids)}")

# ------------------------------------------------------------
# Merge numerator flag back onto denominator population
# ------------------------------------------------------------
denom_df['a1c_compliant'] = denom_df['DESYNPUF_ID'].isin(compliant_ids).astype(int)

compliance_rate = denom_df['a1c_compliant'].mean()
print(f"\n{'='*50}")
print(f"HEDIS A1c TESTING COMPLIANCE RATE: {compliance_rate:.1%}")
print(f"Compliant: {denom_df['a1c_compliant'].sum()} / {len(denom_df)}")
print(f"{'='*50}")

# Save for Step 3 (feature engineering + modeling)
denom_df.to_csv('patients_with_compliance_label.csv', index=False)
print("\nSaved patients_with_compliance_label.csv - ready for Step 3 (feature engineering)")

Denominator patients to check: 1380

Scanning Carrier Claims Sample 1A for A1c tests (this may take a minute)...
Found 678 A1c claim lines in file 1A

Scanning Carrier Claims Sample 1B for A1c tests...
Found 747 A1c claim lines in file 1B

Total A1c claim lines found: 1425

Unique patients with >=1 A1c test in 2010: 372

HEDIS A1c TESTING COMPLIANCE RATE: 27.0%
Compliant: 372 / 1380

Saved patients_with_compliance_label.csv - ready for Step 3 (feature engineering)


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, average_precision_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

DATA_DIR = 'synpuf_sample1'

df = pd.read_csv('patients_with_compliance_label.csv')
denom_ids = set(df['DESYNPUF_ID'])

A1C_CODES = {'83036', '83037'}
HCPCS_COLS = [f'HCPCS_CD_{i}' for i in range(1, 14)]
USECOLS = ['DESYNPUF_ID', 'CLM_FROM_DT'] + HCPCS_COLS

def find_a1c_claims(filepath, ids, chunksize=50000):
    matches = []
    for chunk in pd.read_csv(filepath, usecols=USECOLS, dtype=str, chunksize=chunksize):
        chunk = chunk[chunk['DESYNPUF_ID'].isin(ids)]
        if chunk.empty:
            continue
        has_a1c = chunk[HCPCS_COLS].isin(A1C_CODES).any(axis=1)
        if has_a1c.any():
            matches.append(chunk[has_a1c][['DESYNPUF_ID', 'CLM_FROM_DT']])
    if matches:
        return pd.concat(matches, ignore_index=True)
    return pd.DataFrame(columns=['DESYNPUF_ID', 'CLM_FROM_DT'])

a1c_1a = find_a1c_claims(f'{DATA_DIR}/DE1_0_2008_to_2010_Carrier_Claims_Sample_1A.csv', denom_ids)
a1c_1b = find_a1c_claims(f'{DATA_DIR}/DE1_0_2008_to_2010_Carrier_Claims_Sample_1B.csv', denom_ids)
a1c_all = pd.concat([a1c_1a, a1c_1b], ignore_index=True)
a1c_all['CLM_FROM_DT'] = pd.to_datetime(a1c_all['CLM_FROM_DT'], format='%Y%m%d')
a1c_all['year'] = a1c_all['CLM_FROM_DT'].dt.year

prior_compliant_ids = set(a1c_all[a1c_all['year'].isin([2008, 2009])]['DESYNPUF_ID'].unique())
df['prior_year_a1c_tested'] = df['DESYNPUF_ID'].isin(prior_compliant_ids).astype(int)
print(f'patients with prior-year (2008-2009) A1c test: {df["prior_year_a1c_tested"].sum()} / {len(df)}')

outpatient = pd.read_csv(
    f'{DATA_DIR}/DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv',
    usecols=['DESYNPUF_ID', 'CLM_FROM_DT'],
    dtype={'DESYNPUF_ID': str}
)
outpatient['CLM_FROM_DT'] = pd.to_datetime(outpatient['CLM_FROM_DT'], format='%Y%m%d')
outpatient['year'] = outpatient['CLM_FROM_DT'].dt.year

visits_2010 = outpatient[outpatient['year'] == 2010].groupby('DESYNPUF_ID').size().rename('outpatient_visits_2010')
visits_prior = outpatient[outpatient['year'].isin([2008, 2009])].groupby('DESYNPUF_ID').size().rename('outpatient_visits_prior_years')
last_visit_2010 = outpatient[outpatient['year'] == 2010].groupby('DESYNPUF_ID')['CLM_FROM_DT'].max().rename('last_outpatient_visit')

df = df.merge(visits_2010, left_on='DESYNPUF_ID', right_index=True, how='left')
df = df.merge(visits_prior, left_on='DESYNPUF_ID', right_index=True, how='left')
df = df.merge(last_visit_2010, left_on='DESYNPUF_ID', right_index=True, how='left')
df['outpatient_visits_2010'] = df['outpatient_visits_2010'].fillna(0)
df['outpatient_visits_prior_years'] = df['outpatient_visits_prior_years'].fillna(0)

year_end = pd.Timestamp('2010-12-31')
df['days_since_last_visit'] = (year_end - df['last_outpatient_visit']).dt.days
df['days_since_last_visit'] = df['days_since_last_visit'].fillna(730)

inpatient = pd.read_csv(
    f'{DATA_DIR}/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv',
    usecols=['DESYNPUF_ID', 'CLM_FROM_DT'],
    dtype={'DESYNPUF_ID': str}
)
inpatient['CLM_FROM_DT'] = pd.to_datetime(inpatient['CLM_FROM_DT'], format='%Y%m%d')
inpatient_2010 = inpatient[inpatient['CLM_FROM_DT'].dt.year == 2010]
inpatient_counts = inpatient_2010.groupby('DESYNPUF_ID').size().rename('inpatient_stays_2010')
df = df.merge(inpatient_counts, left_on='DESYNPUF_ID', right_index=True, how='left')
df['inpatient_stays_2010'] = df['inpatient_stays_2010'].fillna(0)

rx = pd.read_csv(
    f'{DATA_DIR}/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv',
    usecols=['DESYNPUF_ID', 'SRVC_DT', 'TOT_RX_CST_AMT'],
    dtype={'DESYNPUF_ID': str}
)
rx['SRVC_DT'] = pd.to_datetime(rx['SRVC_DT'], format='%Y%m%d')
rx_2010 = rx[rx['SRVC_DT'].dt.year == 2010]
rx_fills = rx_2010.groupby('DESYNPUF_ID').size().rename('rx_fills_2010')
rx_cost = rx_2010.groupby('DESYNPUF_ID')['TOT_RX_CST_AMT'].sum().rename('rx_total_cost_2010')

df = df.merge(rx_fills, left_on='DESYNPUF_ID', right_index=True, how='left')
df = df.merge(rx_cost, left_on='DESYNPUF_ID', right_index=True, how='left')
df['rx_fills_2010'] = df['rx_fills_2010'].fillna(0)
df['rx_total_cost_2010'] = df['rx_total_cost_2010'].fillna(0)

carrier_counts = []
for fname in ['DE1_0_2008_to_2010_Carrier_Claims_Sample_1A.csv', 'DE1_0_2008_to_2010_Carrier_Claims_Sample_1B.csv']:
    for chunk in pd.read_csv(f'{DATA_DIR}/{fname}', usecols=['DESYNPUF_ID', 'CLM_FROM_DT'], dtype=str, chunksize=50000):
        chunk = chunk[chunk['DESYNPUF_ID'].isin(denom_ids)]
        if chunk.empty:
            continue
        chunk['CLM_FROM_DT'] = pd.to_datetime(chunk['CLM_FROM_DT'], format='%Y%m%d')
        chunk = chunk[chunk['CLM_FROM_DT'].dt.year == 2010]
        carrier_counts.append(chunk)
carrier_2010 = pd.concat(carrier_counts, ignore_index=True)
carrier_claim_counts = carrier_2010.groupby('DESYNPUF_ID').size().rename('carrier_claims_2010')
df = df.merge(carrier_claim_counts, left_on='DESYNPUF_ID', right_index=True, how='left')
df['carrier_claims_2010'] = df['carrier_claims_2010'].fillna(0)

df['sex'] = df['BENE_SEX_IDENT_CD'].map({1: 'M', 2: 'F'})
df['race'] = df['BENE_RACE_CD'].map({1: 'White', 2: 'Black', 3: 'Other', 5: 'Hispanic'}).fillna('Other')

condition_cols = {
    'SP_CHF': 'heart_failure',
    'SP_CHRNKIDN': 'kidney_disease',
    'SP_ISCHMCHT': 'ischemic_heart_disease',
    'SP_DEPRESSN': 'depression',
    'SP_COPD': 'copd',
    'SP_STRKETIA': 'stroke_history',
}
for src, name in condition_cols.items():
    df[name] = (df[src] == 1).astype(int)

feature_cols = [
    'age_2010', 'prior_year_a1c_tested', 'outpatient_visits_2010',
    'outpatient_visits_prior_years', 'days_since_last_visit',
    'inpatient_stays_2010', 'rx_fills_2010', 'rx_total_cost_2010',
    'carrier_claims_2010', 'heart_failure', 'kidney_disease',
    'ischemic_heart_disease', 'depression', 'copd', 'stroke_history',
]
model_df = df[feature_cols + ['sex', 'race', 'a1c_compliant']].copy()
model_df = pd.get_dummies(model_df, columns=['sex', 'race'], drop_first=True)

X = model_df.drop(columns=['a1c_compliant'])
y = model_df['a1c_compliant']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
print('logistic regression')
print(classification_report(y_test, log_reg.predict(X_test_scaled)))
print('roc auc:', round(roc_auc_score(y_test, y_proba_lr), 3))
print('pr auc:', round(average_precision_score(y_test, y_proba_lr), 3))

rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
print('\nrandom forest')
print(classification_report(y_test, rf.predict(X_test)))
print('roc auc:', round(roc_auc_score(y_test, y_proba_rf), 3))
print('pr auc:', round(average_precision_score(y_test, y_proba_rf), 3))

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
print('\nxgboost')
print(classification_report(y_test, xgb.predict(X_test)))
print('roc auc:', round(roc_auc_score(y_test, y_proba_xgb), 3))
print('pr auc:', round(average_precision_score(y_test, y_proba_xgb), 3))

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print('\ntop features (random forest)')
print(importances.head(10))

baseline_rate = y.mean()
print(f'\nbaseline (always predict majority class) accuracy: {max(baseline_rate, 1-baseline_rate):.3f}')

patients with prior-year (2008-2009) A1c test: 639 / 1380
logistic regression
              precision    recall  f1-score   support

           0       0.81      0.65      0.72       252
           1       0.38      0.59      0.46        93

    accuracy                           0.63       345
   macro avg       0.60      0.62      0.59       345
weighted avg       0.70      0.63      0.65       345

roc auc: 0.672
pr auc: 0.446

random forest
              precision    recall  f1-score   support

           0       0.76      0.75      0.75       252
           1       0.35      0.37      0.36        93

    accuracy                           0.64       345
   macro avg       0.55      0.56      0.55       345
weighted avg       0.65      0.64      0.65       345

roc auc: 0.631
pr auc: 0.381

xgboost
              precision    recall  f1-score   support

           0       0.76      0.70      0.73       252
           1       0.32      0.39      0.35        93

    accuracy          